In [0]:
""" 
Project Overview:
A realistic end-to-end PySpark project that processes raw HR data from multiple sources, cleans and enriches it, runs business analysis, and saves final reports — covering every topic you have learned.
SCENARIO
You are a data engineer at a company. The HR team has given you raw data files in CSV, JSON, and Parquet format. Your job is to build a PySpark pipeline that produces a clean, enriched employee report and answers key business questions.
DATASETS TO USE
Employees.csv :
Main employee data — id, name, dept, salary, join_date, gender, age, is_active, bonus, rating
Departments.json :
dept, location, manager, budget
new_hires.parquet :
Same schema as employees - new joiners to be appended
salary_bands.csv :
dept, gender, band_label 

The project has 7 phases with 26 tasks total:
•	Phase 1 - run once to generate all 4 sample datasets (CSV, JSON, Parquet)
•	Phase 2 - ingest all files and verify schemas
•	Phase 3 - clean and transform (withColumn, filter, drop, rename)
              3.1 Convert join_date to DateType
              3.2  Add experience_level column
                     Add column experience_level: year < 2019 = 'Senior', 2019–2021 = 'Mid', after 2021 = 'Junior'. Use year() function.
              3.3  Add days_employed column
                     Add days_employed = number of days from join_date to today
              3.4   Add salary_band column
                     salary >= 90000 → 'High', salary >= 70000 → 'Mid', else →'Low'.
               3.5 Keep only active employees
                     Filter to keep only rows where is_active = True.
              3.6 Rename and drop columns
                               Rename dept → department. Drop the gender column from df_active.

•	Phase 4 - enrich via joins (inner, left, anti, multi-key, self join)
                 4.1 Inner join with departments
Join df_emp with df_dept on dept. Add location, manager, and budget to each employee row.
                           4.2 Left join to find unmatched depts
Left join df_emp with df_dept. Then show employees whose dept has no match in departments (location is null).
                          4.3 Anti join for missing departments
Use an anti join to find departments in df_dept that have no employees at all.
                          4.4 Join with salary bands on two keys
Join df_emp with df_bands on both dept and gender. Add band_label to each employee.
                          4.5 Chained join (departments + bands)
Join df_emp with both df_dept and df_bands in one chain. Show name, dept, location, manager, band_label.
                          4.6 Self join to find dept colleagues
Self join df_emp to find all unique pairs of employees in the same department. Show emp1, emp2, dept.
•	Phase 5 - business analysis (SQL, GroupBy, Pivot, orderBy)
               5.1 Top 3 highest paid employees (SQL)
Register df_emp as a temp view. Use SQL to find the top 3 highest paid employees by name and salary.
                         5.2 Depts with avg salary above 70000 (SQL)
Using SQL with HAVING, find departments where average salary exceeds 70000.
                         5.3 Dept headcount, avg salary, total bonus
Group by dept. Show headcount, average salary (rounded to 0 decimals), and total bonus per dept. Sort by avg salary desc.
                   
•	Phase 6 - combine with new hires using Union
                 6.1 Union employees with new hires
Use unionByName() to combine df_emp and df_hires. Print the total row count.
                           6.2 Re-apply transformations to full dataset
On df_all, re-apply: convert join_date, add experience_level, add salary_band, add days_employed.
                          6.3 Enrich full dataset with dept info
Join df_all with df_dept on dept. Add location, manager, and budget. Store as df_final.
                        6.4 Select final columns
Select only: id, name, dept, location, manager, salary, salary_band, bonus, experience_level, days_employed, rating.

•	Phase 7 - save outputs in Parquet and CSV, with partitioning
                7.1 Save final report as Parquet
Write df_final to output/final_report.parquet. Overwrite if exists.
                           7.2 Save dept summary as CSV
Save the GroupBy dept summary (from task 4.2) as a CSV with headers to output/dept_summary.
                          7.3 Read back and verify
Read the saved final_report.parquet back. Print its schema, row count, and show first 5 rows.
                         7.4 Save one file per department (partition)
Write df_final partitioned by dept to output/partitioned_report as Parquet.

Sample data:- 
# employees.csv
emp_data = [
  (1,"Alice","Engineering",95000,"2021-03-15","F",29,True,9500,4),
  (2,"Bob","Marketing",72000,"2019-07-01","M",35,True,7200,3),
  (3,"Carol","Engineering",88000,"2020-11-20","F",31,False,8800,5),
  (4,"David","HR",61000,"2022-01-10","M",27,True,6100,4),
  (5,"Eve","Marketing",79000,"2018-05-25","F",38,True,7900,4),
  (6,"Frank","Engineering",102000,"2017-09-30","M",42,False,10200,5),
  (7,"Grace","HR",58000,"2023-02-14","F",24,True,5800,3),
]
emp_schema = StructType([
  StructField("id",IntegerType(),False),
  StructField("name",StringType(),True),
  StructField("dept",StringType(),True),
  StructField("salary",IntegerType(),True),
  StructField("join_date",StringType(),True),
  StructField("gender",StringType(),True),
  StructField("age",IntegerType(),True),
  StructField("is_active",BooleanType(),True),
  StructField("bonus",IntegerType(),True),
  StructField("rating",IntegerType(),True),
])
 
 
# departments.json
dept_data = [
  ("Engineering","New York","Sara",500000),
  ("Marketing","Chicago","Tom",300000),
  ("HR","Austin","Asha",200000),
  ("Finance","Dallas","Leo",400000),
]
dept_schema = StructType([
  StructField("dept",StringType(),True),
  StructField("location",StringType(),True),
  StructField("manager",StringType(),True),
  StructField("budget",IntegerType(),True),
])
 
 
# new_hires.parquet
hire_data = [
  (8,"Heidi","Finance",85000,"2024-01-05","F",28,True,8500,4),
  (9,"Ivan","Engineering",91000,"2023-08-01","M",30,True,9100,5),
  (10,"Judy","Marketing",67000,"2024-03-10","F",26,True,6700,3),
]
 
# salary_bands.csv
band_data = [
  ("Engineering","F","Band-A"),("Engineering","M","Band-A"),
  ("Marketing","F","Band-B"),("Marketing","M","Band-B"),
  ("HR","F","Band-C"),("HR","M","Band-C"),
  ("Finance","F","Band-B"),("Finance","M","Band-B"),
]
band_schema = StructType([
  StructField("dept",StringType(),True),
  StructField("gender",StringType(),True),
  StructField("band_label",StringType(),True),
])

"""

In [0]:
# employees.csv file
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
emp_data = [
  (1,"Alice","Engineering",95000,"2021-03-15","F",29,True,9500,4),
  (2,"Bob","Marketing",72000,"2019-07-01","M",35,True,7200,3),
  (3,"Carol","Engineering",88000,"2020-11-20","F",31,False,8800,5),
  (4,"David","HR",61000,"2022-01-10","M",27,True,6100,4),
  (5,"Eve","Marketing",79000,"2018-05-25","F",38,True,7900,4),
  (6,"Frank","Engineering",102000,"2017-09-30","M",42,False,10200,5),
  (7,"Grace","HR",58000,"2023-02-14","F",24,True,5800,3),
]
emp_schema = StructType([
  StructField("id",IntegerType(),False),
  StructField("name",StringType(),True),
  StructField("dept",StringType(),True),
  StructField("salary",IntegerType(),True),
  StructField("join_date",StringType(),True),
  StructField("gender",StringType(),True),
  StructField("age",IntegerType(),True),
  StructField("is_active",BooleanType(),True),
  StructField("bonus",IntegerType(),True),
  StructField("rating",IntegerType(),True),
])



In [0]:
# departments.json file
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
dept_data = [
  ("Engineering","New York","Sara",500000),
  ("Marketing","Chicago","Tom",300000),
  ("HR","Austin","Asha",200000),
  ("Finance","Dallas","Leo",400000),
]
dept_schema = StructType([
  StructField("dept",StringType(),True),
  StructField("location",StringType(),True),
  StructField("manager",StringType(),True),
  StructField("budget",IntegerType(),True),
])


In [0]:
# new_hires.parquet file
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
hire_data = [
  (8,"Heidi","Finance",85000,"2024-01-05","F",28,True,8500,4),
  (9,"Ivan","Engineering",91000,"2023-08-01","M",30,True,9100,5),
  (10,"Judy","Marketing",67000,"2024-03-10","F",26,True,6700,3),
]

In [0]:
# salary_bands.csv
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType
band_data = [
  ("Engineering","F","Band-A"),("Engineering","M","Band-A"),
  ("Marketing","F","Band-B"),("Marketing","M","Band-B"),
  ("HR","F","Band-C"),("HR","M","Band-C"),
  ("Finance","F","Band-B"),("Finance","M","Band-B"),
]
band_schema = StructType([
  StructField("dept",StringType(),True),
  StructField("gender",StringType(),True),
  StructField("band_label",StringType(),True),
])


In [0]:
"""
df_emp = spark.createDataFrame(emp_data, emp_schema)

- createDataFrame():
    Converts raw data (emp_data) into a structured table (DataFrame)

- emp_data:
    Actual data (list/records)

- emp_schema:
    Column names + data types

- df_emp:
    Variable name holding the DataFrame (like storing table in memory)


df_emp.write.mode("overwrite").csv("/Volumes/dataframeassigment/project_1/project_one/employees", header=True)

- write:
    Used to save DataFrame to storage

- mode("overwrite"):
    If data already exists → delete and replace with new data

- csv():
    Saves the DataFrame in CSV format

- "/Volumes/...":
    Path where data is stored (Unity Catalog Volume)

- header=True:
    Includes column names in the first row of CSV
"""

In [0]:
#                The project has 7 phases with 26 tasks total:

In [0]:
#   Phase 1 - run once to generate all 4 sample datasets (CSV, JSON, Parquet)

In [0]:
# employees csv
df_emp = spark.createDataFrame(emp_data,emp_schema)
df_emp.write.mode("overwrite").csv("/Volumes/dataframeassigment/project_1/project_one/employees", header=True)

# departments json
df_dept = spark.createDataFrame(dept_data,dept_schema)
df_dept.write.mode("overwrite").json("/Volumes/dataframeassigment/project_1/project_one/departments")

# hires Parquet
df_hires = spark.createDataFrame(hire_data, emp_schema)
df_hires.write.mode("overwrite").parquet("/Volumes/dataframeassigment/project_1/project_one/hires")

# salary bands csv
df_bands = spark.createDataFrame(band_data, band_schema)
df_bands.write.mode("overwrite").csv("/Volumes/dataframeassigment/project_1/project_one/salary_bands", header=True)

In [0]:
#   Phase 2 - ingest all files and verify schemas
"""
Ingest - Read data from files into DataFrames

Verify schema - Check structure of DataFrame (column names + data types)

inferSchema=True = automatically detect column data types instead of treating everything as string
"""

In [0]:
# Read employees (CSV)
df_emp = spark.read.csv("/Volumes/dataframeassigment/project_1/project_one/employees", header=True, inferSchema=True)

# Read departments (JSON)
df_dept = spark.read.json("/Volumes/dataframeassigment/project_1/project_one/departments")

# Read hires (Parquet)
df_hires = spark.read.parquet("/Volumes/dataframeassigment/project_1/project_one/hires")

# Read salary bands (CSV)
df_bands = spark.read.csv("/Volumes/dataframeassigment/project_1/project_one/salary_bands",header=True,inferSchema=True)

In [0]:
print("df_emp:")
df_emp.printSchema()
print("df_dept:")
df_dept.printSchema()
print("df_hires:")
df_hires.printSchema()
print("df_bands:")
df_bands.printSchema()


In [0]:
"""
CSV → dumb file → needs help  
JSON → smart file → knows structure  
Parquet → very smart → stores schema

header=True → tells Spark column names (CSV only)
inferSchema=True → detects data types (CSV only)
Not needed for JSON & Parquet because they already store structure

"""

In [0]:
#   Phase 3 - clean and transform (withColumn, filter, drop, rename)

In [0]:
#   3.1 Convert join_date to DateType

from pyspark.sql.functions import col, to_date
print("Before casting join_date: String()")

df_emp.withColumn("join_date", to_date(col("join_date"), "dd-MM-yyyy")).show()

print("After casting join_date: ")
print(df_emp.schema["join_date"].dataType)

In [0]:
#3.2  Add experience_level column --- Add column experience_level: year < 2019 = 'Senior', 2019–2021 = 'Mid', after 2021 = 'Junior'. Use year() function.


from pyspark.sql.functions import col, year, when
df_emp.withColumn("experince_level", when(year(col("join_date"))<2019, "Senior").when((year(col("join_date"))>=2019) & (year(col("join_date"))<=2021), "Mid").otherwise("Junior")).show()


In [0]:
# 3.3  Add days_employed column -- Add days_employed = number of days from join_date to today

from pyspark.sql.functions import col, current_date, datediff
df_emp.withColumn("days_employed", datediff(current_date(), col("join_date"))).display()

In [0]:
# 3.4   Add salary_band column -- salary >= 90000 → 'High', salary >= 70000 → 'Mid', else →'Low'.

from pyspark.sql.functions import col, when
df_emp.withColumn("Salary_Band", when(col("salary")>=90000, "High") .when(col("salary")>=70000, "Mid") .otherwise("Low")).display()

In [0]:
# 3.5 Keep only active employees --- Filter to keep only rows where is_active = True.

from pyspark.sql.functions import col
df_active = df_emp.filter(col("is_active") == True)
df_active.show()

In [0]:
# 3.6 Rename and drop columns -- Rename dept → department. Drop the gender column from df_active.

df_active.withColumnRenamed("dept", "department").drop("gender").show()

In [0]:
#   Phase 4 - enrich via joins (inner, left, anti, multi-key, self join)

In [0]:
# 4.1 Inner join with departments -- Join df_emp with df_dept on dept. Add location, manager, and budget to each employee row.

df_innerJoin = df_emp.join(df_dept, on = "dept", how="inner")
df_innerJoin.display()

In [0]:
# left join
df_emp.join(df_dept, on = "dept", how="left").display()

In [0]:
# 4.2 Left join to find unmatched depts -- Left join df_emp with df_dept. Then show employees whose dept has no match in departments (location is null).

from pyspark.sql.functions import col
df_emp.join(df_dept, on = "dept", how="left").filter(col("location").isNull()).display()

In [0]:
# 4.3 Anti join for missing departments -- Use an anti join to find departments in df_dept that have no employees at all.

from pyspark.sql.functions import col
df_dept.join(df_emp, on = "dept", how="left_anti").display()

In [0]:
# 4.4 Join with salary bands on two keys -- Join df_emp with df_bands on both dept and gender. Add band_label to each employee.

df_bandJoin = df_emp.join(df_bands, on = ["dept", "gender"], how="inner")
df_bandJoin.display()

In [0]:
# 4.5 Chained join (departments + bands) -- Join df_emp with both df_dept and df_bands in one chain. Show name, dept, location, manager, band_label.


df_chain = df_emp.join(df_dept, on ="dept", how="inner").join(df_bands, on = ["dept", "gender"], how="inner").select("name", "dept", "location", "manager", "band_label")
df_chain.display()

In [0]:
# 4.6 Self join to find dept colleagues -- Self join df_emp to find all unique pairs of employees in the same department. Show emp1, emp2, dept.


from pyspark.sql.functions import col

emp1 = df_emp.alias("emp1")
emp2 = df_emp.alias("emp2")

df_pairs = emp1.join(
    emp2,
    (col("emp1.dept") == col("emp2.dept")) &
    (col("emp1.id") < col("emp2.id")),
    "inner"
).select(
    col("emp1.name").alias("emp1"),
    col("emp2.name").alias("emp2"),
    col("emp1.dept").alias("dept")
)

df_pairs.display()

In [0]:
#   	Phase 5 - business analysis (SQL, GroupBy, Pivot, orderBy)

In [0]:
# 5.1 Top 3 highest paid employees (SQL) -- Register df_emp as a temp view. Use SQL to find the top 3 highest paid employees by name and salary.


df_emp.createOrReplaceTempView("emp")
Top_3_paid_emp = spark.sql("""
select name, salary from emp 
order by salary desc 
limit 3 
""")
Top_3_paid_emp.display()

In [0]:
# 5.2 Depts with avg salary above 70000 (SQL) -- Using SQL with HAVING, find departments where average salary exceeds 70000.


df_emp.createOrReplaceTempView("emp")
dept_avg_salary = spark.sql(""" 
select dept, avg(salary) as avg_sal from emp
group by dept 
having avg(salary) > 70000
""")
dept_avg_salary.display()


In [0]:
# 5.3 Dept headcount, avg salary, total bonus -- Group by dept. Show headcount, average salary (rounded to 0 decimals), and total bonus per dept. Sort by avg salary desc.


df_emp.createOrReplaceTempView("emp")
df_summary = spark.sql("""
select dept, count(*) as headcount, round(avg(salary),0) as avg_salary, sum(bonus) as total_bonus from emp
group by dept 
sort by avg_salary desc                  
""")
df_summary.display()

In [0]:
#       Phase 6 - combine with new hires using Union

In [0]:
# 6.1 Union employees with new hires -- Use unionByName() to combine df_emp and df_hires. Print the total row count.

df_all = df_emp.unionByName(df_hires)
df_all.display()
print("total number of rows: ", df_all.count())


In [0]:
# 6.2 Re-apply transformations to full dataset -- On df_all, re-apply: convert join_date, add experience_level, add salary_band, add days_employed.

from pyspark.sql.functions import col, datediff, current_date, when

df_transformed = (
    df_all
    .withColumn("join_date", col("join_date").cast("date"))
    .withColumn("days_employed", datediff(current_date(), col("join_date")))
    .withColumn(
        "experience_level",
        when(col("days_employed") < 365, "Fresher")
        .when(col("days_employed") < 2000, "Mid")
        .otherwise("Senior")
    )
    .withColumn(
        "salary_band",
        when(col("salary") < 50000, "Low")
        .when(col("salary") < 100000, "Medium")
        .otherwise("High")
    )
)

df_transformed.display()

In [0]:
#  6.3 Enrich full dataset with dept info -- Join df_all with df_dept on dept. Add location, manager, and budget. Store as df_final.

df_final = df_all.join(df_dept, "dept", "left")
df_final.display()

In [0]:
# 6.4 Select final columns -- Select only: id, name, dept, location, manager, salary, salary_band, bonus, experience_level, days_employed, rating.

from pyspark.sql.functions import col

df_final_Columns = df_transformed.join(df_dept, "dept", "left").select("id", "name", "dept", "location", "manager","salary", "salary_band", "bonus","experience_level", "days_employed", "rating")

df_final_Columns.display()

In [0]:
#      Phase 7 - save outputs in Parquet and CSV, with partitioning

In [0]:
# 7.1 Save final report as Parquet -- Write df_final to output/final_report.parquet. Overwrite if exists.


df_final_Columns.write.mode("overwrite").parquet("/Volumes/dataframeassigment/project_1/project_one/output/final_report.parquet")

In [0]:
# 7.2 Save dept summary as CSV -- Save the GroupBy dept summary (from task 4.2) as a CSV with headers to output/dept_summary. The actual groupBy dept summary is in task 5.3 (Cell 32)

df_summary.write.mode("overwrite").csv("/Volumes/dataframeassigment/project_1/project_one/output/dept_summary", header = True)


In [0]:
# 7.3 Read back and verify -- Read the saved final_report.parquet back. Print its schema, row count, and show first 5 rows.
df_final_read = spark.read.parquet("/Volumes/dataframeassigment/project_1/project_one/output/final_report.parquet")
df_final_read.printSchema()
print("total number of rows: ", df_final_read.count())
display(df_final_read.limit(5))

In [0]:
# 7.4 Save one file per department (partition) -- Write df_final partitioned by dept to output/partitioned_report as Parquet.

df_final_Columns.write.mode("overwrite").partitionBy("dept").parquet("/Volumes/dataframeassigment/project_1/project_one/output/partitioned_report")
